# 01 — SDK Schema, Q1 Check, Q2 Diagnose

This notebook walks the foundation of the FactPy v0.1 round story:

1. Author a `Person` schema with the `kernel.sdk` declarative DSL.
2. Compile the schema, build a `Store`, and seed four people directly through the low-level write protocol so the demo can show every artifact id (`asrt_id`, `pred_id`) the application layer consumes.
3. Compile a derivation plan whose body is `Person.exists($p) ∧ age($p, $age) ∧ region($p, $region)`.
4. **Q1 Check** — `check_derivation_binding(...)` confirms a specific binding is supported by the ledger.
5. **Q2 Diagnose** — `diagnose_derivation_binding(...)` localizes the responsible body atom for a binding that does *not* hold.

Every cell below imports real APIs from `kernel.application` / `kernel.sdk` and asserts on the structured result. There is no demo helper to import — what you read is what you would write.

**Prerequisites:** None. **Next:** [02_overlay_why_not_frontier.ipynb](02_overlay_why_not_frontier.ipynb) — Q3 Fact Overlay, Q4 Why-not Universe, Q5 Frontier Trace.

## Setup

Inject the repository `src/` directory onto `sys.path` so the notebook works whether you launched Jupyter from the repo root or from `examples/`.

In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
_src_dir = _repo_root / 'src'
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from kernel.application import (
    build_schema_index,
    check_derivation_binding,
    diagnose_derivation_binding,
    entity_info,
    field_predicate,
    resolve_selector,
)
from kernel.application.protocol import (
    CheckRequest,
    CompiledDerivationPlan,
    CompiledHeadCall,
    DiagnoseRequest,
    EntitySelector,
)
from kernel.core.evidence.write_protocol import set_field
from kernel.core.store import Store
from kernel.sdk import Entity, Field, Identity, compile_schema_from_classes

## 1. Author a `Person` schema

Schemas are plain Python classes deriving from `kernel.sdk.Entity`. `Identity(primary_key=True)` marks the identity field; `Field(cardinality='single')` declares a single-valued attribute.

`compile_schema_from_classes(...)` returns a `SchemaIR` (the compiled, predicate-flat representation the kernel operates on). `build_schema_index(...)` wraps it in the lookup helper used by the application layer.

In [ ]:
class Person(Entity):
    name: str = Identity(primary_key=True)
    age: int = Field(cardinality='single')
    region: str = Field(cardinality='single')


schema_ir = compile_schema_from_classes([Person])
index = build_schema_index(schema_ir)

person_info = entity_info(index, 'Person')
age_pred = field_predicate(index, 'Person', 'age').pred_id
region_pred = field_predicate(index, 'Person', 'region').pred_id

print(f'Person.exists predicate id : {person_info.exists_predicate_id!r}')
print(f'Person.age    predicate id : {age_pred!r}')
print(f'Person.region predicate id : {region_pred!r}')

## 2. Build a Store and seed four people

The `Store` owns an in-memory ledger. We seed four people (`alice` 25/us, `bob` 30/eu, `carol` 28/us, `dave` 17/us) by calling `set_field(...)` directly so the demo can capture every `asrt_id` the application capabilities will reference later (Diagnose, Fact Overlay, ProofFrame all use them).

`resolve_selector(...)` turns an `EntitySelector` (entity type + identity) into the canonical `e_ref` that all per-entity predicates key on.

In [ ]:
@dataclass(frozen=True)
class SeededPerson:
    e_ref: str
    age: int
    region: str
    exists_asrt_id: str
    age_asrt_id: str
    region_asrt_id: str


def seed_person(store: Store, *, name: str, age: int, region: str) -> SeededPerson:
    ref = resolve_selector(
        EntitySelector(entity_type='Person', identity={'name': name}),
        index=index,
    )
    info = entity_info(index, 'Person')
    encoded = ref.encoded_ref or ''
    exists_asrt_id = set_field(store.ledger, info.exists_predicate_id, encoded, [])
    name_pred_id = info.identity_predicates['name'].pred_id
    set_field(store.ledger, name_pred_id, encoded, [('string', name)])
    age_asrt_id = set_field(
        store.ledger,
        field_predicate(index, 'Person', 'age').pred_id,
        encoded,
        [('int', age)],
    )
    region_asrt_id = set_field(
        store.ledger,
        field_predicate(index, 'Person', 'region').pred_id,
        encoded,
        [('string', region)],
    )
    return SeededPerson(
        e_ref=encoded,
        age=age,
        region=region,
        exists_asrt_id=exists_asrt_id,
        age_asrt_id=age_asrt_id,
        region_asrt_id=region_asrt_id,
    )


store = Store(schema_ir)
people = {
    'alice': seed_person(store, name='alice', age=25, region='us'),
    'bob':   seed_person(store, name='bob',   age=30, region='eu'),
    'carol': seed_person(store, name='carol', age=28, region='us'),
    'dave':  seed_person(store, name='dave',  age=17, region='us'),
}

for name, person in people.items():
    print(f'  {name:5s}  e_ref={person.e_ref}  age={person.age}  region={person.region!r}')

## 3. Compile a derivation plan

A `CompiledDerivationPlan` is the compiled form a derivation in v0.1: a list of body atoms and a head call describing which variables are projected.

Below we declare a single-branch body of three predicates — entity existence, age binding, region binding — and a head that projects all three variables. This plan is the input to every Q1–Q5 capability in chapters 1–2.

In [ ]:
plan = CompiledDerivationPlan(
    derivation_id='round-story-person-snapshot',
    version='1.0',
    body_ir=[
        ('pred', person_info.exists_predicate_id, ['$p']),
        ('pred', age_pred,    ['$p', '$age']),
        ('pred', region_pred, ['$p', '$region']),
    ],
    heads=(
        CompiledHeadCall(
            target_pred_id=person_info.exists_predicate_id,
            head_var_names=('$p', '$age', '$region'),
        ),
    ),
)

print(f'derivation_id : {plan.derivation_id!r}')
print(f'body atoms    : {len(plan.body_ir)}')
print(f'head vars     : {plan.heads[0].head_var_names}')

## 4. Q1 Check — does Alice's binding pass?

`check_derivation_binding(CheckRequest(plan, binding, engine))` evaluates the plan against an explicit binding tuple and returns one of `passed | failed | unsupported | invalid_request`.

Bindings are sorted `(var_name, value)` tuples. We use a small `_binding(...)` helper to make the canonical form explicit.

A `passed` result carries `matched_count`, `matched_binding`, and an `evidence_envelope` whose `engine_payload` is the `SupportArtifact` (the per-atom proof witness chapters 3–4 will inspect).

In [ ]:
def _binding(*items):
    return tuple(sorted(items, key=lambda item: item[0]))


alice = people['alice']
binding_alice = _binding(
    ('$p', alice.e_ref),
    ('$age', 25),
    ('$region', 'us'),
)

result = check_derivation_binding(
    CheckRequest(plan=plan, binding=binding_alice, engine='native'),
    store=store,
)

assert result.status == 'passed', result
assert result.matched_count == 1, result
assert result.matched_binding == binding_alice, result
assert result.evidence_envelope is not None

print(f'status              : {result.status}')
print(f'matched_count       : {result.matched_count}')
print(f'matched_binding     : {result.matched_binding}')
print(f'evidence_envelope   : {type(result.evidence_envelope).__name__}')
print(f'  engine_payload    : {type(result.evidence_envelope.engine_payload).__name__}')

## 5. Q1 Check — what if we ask for the wrong age?

Same call, same plan, but the binding asserts `$age=99`. The native engine finds no row matching `(p=alice, age=99, region=us)`, so the result is `failed`. Q1 Check tells us *whether* a binding is supported; Q2 Diagnose tells us *where* the support breaks.

In [ ]:
binding_alice_wrong_age = _binding(
    ('$p', alice.e_ref),
    ('$age', 99),
    ('$region', 'us'),
)

result_failed = check_derivation_binding(
    CheckRequest(plan=plan, binding=binding_alice_wrong_age, engine='native'),
    store=store,
)

assert result_failed.status == 'failed', result_failed
print(f'status        : {result_failed.status}')
print(f'matched_count : {result_failed.matched_count}')

## 6. Q2 Diagnose — where does the failing binding break?

`diagnose_derivation_binding(DiagnoseRequest(plan, binding, engine))` runs the plan in a probe mode that walks each body atom and reports the first one that does not match.

Expected for `(p=alice, age=99, region=us)`: `status=failed`, `failure_kind=atom_localized`, `branch_index=0`, `failed_atom_index=1` — the body order is `exists ∧ age ∧ region`, and Alice does exist, so the failure is at atom index 1 (the age binding).

In [ ]:
diagnose_result = diagnose_derivation_binding(
    DiagnoseRequest(plan=plan, binding=binding_alice_wrong_age, engine='native'),
    store=store,
)

assert diagnose_result.status == 'failed'
assert diagnose_result.failure_kind == 'atom_localized'
locator = diagnose_result.diagnostic_payload
assert locator is not None
assert locator.branch_index == 0
assert locator.failed_atom_index == 1

print(f'status              : {diagnose_result.status}')
print(f'failure_kind        : {diagnose_result.failure_kind}')
print(f'branch_index        : {locator.branch_index}')
print(f'failed_atom_index   : {locator.failed_atom_index}  (0=exists, 1=age, 2=region)')
print(f'attempted_binding   : {locator.attempted_binding}')

## Wrap-up

| Capability | API | Module |
|---|---|---|
| Q1 Check    | `check_derivation_binding(CheckRequest)`       | `kernel.application.derivation_check_runtime` |
| Q2 Diagnose | `diagnose_derivation_binding(DiagnoseRequest)` | `kernel.application.diagnose_runtime` |

**Public-surface boundary (Batch 8):** Both runtimes are *advanced-importable* from `kernel.application`. v0.1 ships **no** dedicated SDK shells or service routes for them — the import you see above is the supported path.

**Next:** [02_overlay_why_not_frontier.ipynb](02_overlay_why_not_frontier.ipynb) — Q3 Fact Overlay (what-if without writing the ledger), Q4 Why-not Universe (green/red board over an explicit candidate set), Q5 Evaluator Frontier Trace (substrate-layer view).

For the integrated end-to-end script (run as `python examples/round_story_full_demo.py`), see `round_story_full_demo.py` — the unittest in `src/kernel/tests/test_examples_round_story_full_demo.py` asserts on its `EXPECTED_PHASE_SUMMARY` contract.